# 01 · Warm-up problems

Twenty classic warm-up questions (the ones that open a coding screen), each with a clean solution, the edge-case tests an interviewer would try, and its complexity. Afterwards you can solve any of them out loud in under five minutes, and explain why the naive version is slow.

**Time:** ~60 min · **Runs:** offline on CPU in < 30 s · **Needs:** [Control flow and loops](../01_python_basics/05_control_flow_and_loops.ipynb), [Functions](../01_python_basics/06_functions_args_scope_lambda.ipynb)

## Why this matters in interviews

- Warm-ups fill the first ten minutes, and the interviewer is watching *how* you work: clarifying questions, edge cases, tests. Knowing FizzBuzz is not what they are grading.
- Many rounds go "solve it with the built-in", then "now without it". You need both the Pythonic one-liner and the loop underneath it.
- Even AI-engineer screens open with one of these as a quick filter before the system-design discussion.
- Every problem hides a follow-up: recursion limits, overflow in other languages, unicode, a hidden O(n²).

## What interviewers ask

- "Write FizzBuzz." Then: "add a rule for 7 without another `elif`."
- "Reverse a string. Now without slicing."
- "Is this a palindrome if we ignore punctuation and case?"
- "Fibonacci: why is the recursive version slow, and how do you fix it?"
- "Is `n` prime? Now list every prime below a million."
- "Find the second largest number" / "remove duplicates but keep the order."
- "Find the missing number in 1..n using O(1) extra space."
- "Flatten an arbitrarily nested list."
- Always: "What is the time and space complexity? What are the edge cases?"

## How to work each problem

**In plain English:** interviewers grade the process as much as the answer. For every problem:

1. **Clarify.** Inputs and types, empty input, one element, duplicates, negatives, non-ASCII text. Say your assumptions out loud.
2. **Brute force first.** One sentence plus its complexity, even if you never code it.
3. **Optimise.** Name the trick: a hash set, two pointers, a maths identity, memoisation.
4. **Test.** Run the edge cases before the interviewer asks for them.

Each problem below has a statement, then a solution cell, then a test cell (`assert`s: silence means pass), then one line on complexity. To practise, write your own function, add it to the tuple the test cell loops over, and run the tests.

In [ ]:
import calendar
import dis
import functools
import math
import random
import sys
import time
from collections import Counter

import matplotlib.pyplot as plt

random.seed(0)
plt.rcParams.update({                      # quiet chart style; colour-blind-safe colours below
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#c3c2b7",
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "axes.titlesize": 11, "legend.frameon": False,
})
BLUE, ORANGE = "#2a78d6", "#eb6834"


def raises(exc_type, fn, *args, **kwargs) -> bool:
    """True if fn(*args) raises exc_type (a tiny stand-in for pytest.raises)."""
    try:
        fn(*args, **kwargs)
    except exc_type:
        return True
    return False


print(f"Python {sys.version.split()[0]}; recursion limit {sys.getrecursionlimit()}")

## 1 · Loops and conditionals

### Problem 1 — FizzBuzz

For 1..n return strings: multiples of 3 → `"Fizz"`, of 5 → `"Buzz"`, of both → `"FizzBuzz"`, otherwise the number.
Example: `fizzbuzz(5)` → `['1', '2', 'Fizz', '4', 'Buzz']`.

In [ ]:
def fizzbuzz(n: int) -> list[str]:
    out = []
    for i in range(1, n + 1):
        if i % 15 == 0:                    # test the combined case first
            out.append("FizzBuzz")
        elif i % 3 == 0:
            out.append("Fizz")
        elif i % 5 == 0:
            out.append("Buzz")
        else:
            out.append(str(i))
    return out


def fizzbuzz_rules(n: int, rules=((3, "Fizz"), (5, "Buzz"))) -> list[str]:
    """Table-driven: adding (7, 'Bazz') is a data change, not another elif."""
    return ["".join(word for d, word in rules if i % d == 0) or str(i) for i in range(1, n + 1)]


print(fizzbuzz(15))

In [ ]:
for f in (fizzbuzz, fizzbuzz_rules):
    assert f(0) == [] and f(-3) == []                  # empty / negative n
    assert f(1) == ["1"]                               # single
    assert f(5) == ["1", "2", "Fizz", "4", "Buzz"]
    assert f(15)[-1] == "FizzBuzz"
    assert f(100).count("FizzBuzz") == 6               # 15, 30, 45, 60, 75, 90
assert fizzbuzz_rules(21, ((3, "Fizz"), (5, "Buzz"), (7, "Bazz")))[-1] == "FizzBazz"
print("fizzbuzz: all tests passed")

**Complexity:** O(n) time, O(n) space for the output. The table-driven version answers the follow-up "add 7 → Bazz" with a one-line change; the `if` chain would need every combination.

### Problem 2 — Leap year

Gregorian rule: divisible by 4, except century years, unless divisible by 400.
Example: `2024 → True`, `1900 → False`, `2000 → True`.

In [ ]:
def is_leap(year: int) -> bool:
    return year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)


print({y: is_leap(y) for y in (1900, 2000, 2023, 2024, 2100)})

In [ ]:
assert is_leap(2024) and is_leap(2000) and is_leap(1600)
assert not is_leap(1900) and not is_leap(2023) and not is_leap(2100)
assert is_leap(0) and is_leap(-4)                      # Python's % stays non-negative for a positive divisor
assert all(is_leap(y) == calendar.isleap(y) for y in range(-400, 3001))   # agree with the stdlib
print("is_leap: all tests passed")

**Complexity:** O(1). The trap is the 400 rule (1900 is not a leap year, 2000 is). In real code, say you would call `calendar.isleap`.

### Problem 3 — Swap two variables without a temp

Swap `a` and `b` without a third variable. Example: `a, b = 3, 7` → `a == 7, b == 3`.

In [ ]:
def swap_pythonic(a, b):
    a, b = b, a                    # right side is evaluated first, then unpacked
    return a, b


def swap_arithmetic(a: int, b: int):
    a = a + b
    b = a - b                      # (a + b) - b = old a
    a = a - b                      # (a + b) - old a = old b
    return a, b


def swap_xor(a: int, b: int):
    a ^= b
    b ^= a                         # b ^ (a ^ b) = old a
    a ^= b                         # (a ^ b) ^ old a = old b
    return a, b


def _just_swap(a, b):
    a, b = b, a


ops = [ins.opname for ins in dis.get_instructions(_just_swap)]
print("bytecode for `a, b = b, a`:", ops)

In [ ]:
for swap in (swap_pythonic, swap_arithmetic, swap_xor):
    assert swap(3, 7) == (7, 3)
    assert swap(-5, 12) == (12, -5)            # negatives
    assert swap(4, 4) == (4, 4)                # equal values
    assert swap(0, 2**100) == (2**100, 0)      # Python ints never overflow
assert swap_pythonic("x", [1]) == ([1], "x")   # only unpacking works for any type
assert raises(TypeError, swap_xor, 1.5, 2.5)   # XOR is defined for ints only
assert "BUILD_TUPLE" not in ops                # CPython swaps on the stack; no tuple is built
print("swap: all tests passed")

**Complexity:** O(1). Say that the arithmetic trick overflows in fixed-width languages (C, Java) and XOR only works on integers, which is why in Python you just write `a, b = b, a`.

## 2 · Strings

### Problem 4 — Reverse a string (three ways)

`"hello"` → `"olleh"`. Show slicing, `reversed()` + `join`, and a manual two-pointer loop.

In [ ]:
def reverse_slice(s: str) -> str:
    return s[::-1]


def reverse_builtin(s: str) -> str:
    return "".join(reversed(s))


def reverse_loop(s: str) -> str:
    chars = list(s)                            # strings are immutable, so work on a list
    i, j = 0, len(chars) - 1
    while i < j:                               # swap towards the middle
        chars[i], chars[j] = chars[j], chars[i]
        i, j = i + 1, j - 1
    return "".join(chars)


print(reverse_slice("hello"), reverse_builtin("hello"), reverse_loop("hello"))
print("reversing a flag:", "🇬🇧", "->", reverse_slice("🇬🇧"))

In [ ]:
for rev in (reverse_slice, reverse_builtin, reverse_loop):
    assert rev("") == ""                       # empty
    assert rev("a") == "a"                     # single
    assert rev("hello") == "olleh"
    assert rev("abba") == "abba"               # palindrome
    assert rev("héllo 😀") == "😀 olléh"        # Python strings are code points, not bytes
    assert rev("🇬🇧") == "🇧🇬"                  # a flag is TWO code points: reversing swaps them
    assert rev("café") == "́efac"   # a combining accent detaches from its letter
print("reverse: all tests passed")

**Complexity:** O(n) time and O(n) space for all three (a new string is unavoidable). Avoid `out = ch + out` in a loop: it copies the string every step, which is O(n²). Reversing *user-perceived* characters (flags, accents) needs grapheme clusters, e.g. the third-party `regex` module's `\X`.

### Problem 5 — Palindrome ignoring case and punctuation

Only letters and digits count, case-insensitive.
Example: `"A man, a plan, a canal: Panama"` → `True`; `"race a car"` → `False`.

In [ ]:
def is_palindrome(s: str) -> bool:
    cleaned = [ch.casefold() for ch in s if ch.isalnum()]
    return cleaned == cleaned[::-1]


def is_palindrome_two_pointers(s: str) -> bool:
    """O(1) extra space: skip non-alphanumerics from both ends."""
    i, j = 0, len(s) - 1
    while i < j:
        if not s[i].isalnum():
            i += 1
        elif not s[j].isalnum():
            j -= 1
        elif s[i].casefold() != s[j].casefold():
            return False
        else:
            i, j = i + 1, j - 1
    return True


print(is_palindrome("A man, a plan, a canal: Panama"), is_palindrome("race a car"))

In [ ]:
for f in (is_palindrome, is_palindrome_two_pointers):
    assert f("") and f("a") and f("!!")                # empty, single, nothing alphanumeric
    assert f("A man, a plan, a canal: Panama")
    assert f("No 'x' in Nixon")
    assert not f("race a car")
    assert f("12321") and not f("1a2")                 # digits count
    assert not f("Ésope reste ici et se repose")       # É is not e: strip accents first (02 · normalise)
print("palindrome: all tests passed")

**Complexity:** O(n) time for both; O(n) extra space for the cleaned list, O(1) for two pointers. Use `casefold()` rather than `lower()` for caseless matching (`"ß".casefold() == "ss"`).

### Problem 6 — Count vowels

Count a, e, i, o, u (case-insensitive). Example: `"Hello World"` → `3`.

In [ ]:
VOWELS = frozenset("aeiou")


def count_vowels(s: str) -> int:
    return sum(ch in VOWELS for ch in s.casefold())    # True counts as 1


def count_vowels_by_letter(s: str) -> dict[str, int]:
    counts = Counter(ch for ch in s.casefold() if ch in VOWELS)
    return {v: counts[v] for v in "aeiou"}


print(count_vowels("Hello World"), count_vowels_by_letter("Programming in Python"))

In [ ]:
assert count_vowels("") == 0                           # empty
assert count_vowels("a") == 1                          # single
assert count_vowels("xyz") == 0                        # 'y' is not a vowel here (clarify!)
assert count_vowels("AEIOU aeiou") == 10
assert count_vowels("Hello World") == 3
assert count_vowels("café") == 1                       # 'é' is not in the set: decide up front
assert count_vowels_by_letter("banana") == {"a": 3, "e": 0, "i": 0, "o": 0, "u": 0}
print("count_vowels: all tests passed")

**Complexity:** O(n) time, O(1) space. The questions to ask first: does `y` count, and do accented vowels count?

### Problem 7 — Character frequency

Count every character. Example: `"banana"` → `{'b': 1, 'a': 3, 'n': 2}`; most common → `('a', 3)`.

In [ ]:
def char_freq_manual(s: str) -> dict[str, int]:
    freq = {}
    for ch in s:
        freq[ch] = freq.get(ch, 0) + 1
    return freq


def char_freq(s: str) -> Counter:
    return Counter(s)


print(char_freq_manual("banana"), char_freq("banana").most_common(1))

In [ ]:
for f in (char_freq_manual, char_freq):
    assert f("") == {}                                 # empty
    assert f("a") == {"a": 1}                          # single
    assert f("banana") == {"b": 1, "a": 3, "n": 2}     # duplicates
    assert f("naïve😀😀")["😀"] == 2                     # unicode
    assert f("aA") == {"a": 1, "A": 1}                 # case-sensitive unless you fold
assert list(char_freq_manual("banana")) == ["b", "a", "n"]   # dicts keep first-seen order
assert char_freq("banana").most_common(1) == [("a", 3)]
print("char_freq: all tests passed")

**Complexity:** O(n) time, O(k) space for k distinct characters. `Counter` is the idiomatic answer; the manual loop shows you know what it does.

### Problem 8 — Anagram check

Same letters, same counts, ignoring case and spaces.
Example: `("Listen", "Silent")` → `True`; `("Dormitory", "dirty room")` → `True`; `("aab", "abb")` → `False`.

In [ ]:
def _letters(s: str) -> str:
    return "".join(s.casefold().split())               # fold case, drop all whitespace


def is_anagram_sort(a: str, b: str) -> bool:
    return sorted(_letters(a)) == sorted(_letters(b))  # O(n log n)


def is_anagram_count(a: str, b: str) -> bool:
    return Counter(_letters(a)) == Counter(_letters(b))  # O(n)


print(is_anagram_count("Dormitory", "dirty room"), is_anagram_count("aab", "abb"))

In [ ]:
for f in (is_anagram_sort, is_anagram_count):
    assert f("", "") and f("a", "a")                   # empty, single
    assert f("Listen", "Silent")
    assert f("Dormitory", "dirty room")
    assert not f("ab", "a")                            # different lengths
    assert not f("aab", "abb")                         # same letters, different counts
    assert f("résumé", "mérésu")                       # unicode letters
assert set("aab") == set("abb")                        # ...which is why set() is the WRONG tool
print("anagram: all tests passed")

**Complexity:** sorting is O(n log n) time; counting is O(n) time and O(k) space. The classic wrong answer is `set(a) == set(b)`: it forgets duplicates.

## 3 · Numbers and maths

### Problem 9 — Sum of digits

Example: `1234` → `10`; negatives use the absolute value: `-1234` → `10`.

In [ ]:
def digit_sum_str(n: int) -> int:
    return sum(int(d) for d in str(abs(n)))


def digit_sum_math(n: int) -> int:
    n, total = abs(n), 0
    while n:
        n, d = divmod(n, 10)                           # peel off the last digit
        total += d
    return total


print(digit_sum_str(1234), digit_sum_math(-1234))

In [ ]:
for f in (digit_sum_str, digit_sum_math):
    assert f(0) == 0                                   # zero (the while loop never runs)
    assert f(7) == 7                                   # single digit
    assert f(1234) == 10 and f(-1234) == 10            # negatives
    assert f(999) == 27
    assert f(10**100) == 1                             # big ints are fine
assert all(digit_sum_str(n) == digit_sum_math(n) for n in range(-1000, 1001))
print("digit_sum: all tests passed")

**Complexity:** O(d) time where d = number of digits = O(log n). The string version is shorter; the `divmod` version is what "without converting to a string" means.

### Problem 10 — Armstrong number

A number equal to the sum of its digits, each raised to the number of digits: `153 = 1³ + 5³ + 3³`.
Example: `153 → True`, `154 → False`; all of them below 1000 → `0..9, 153, 370, 371, 407`.

In [ ]:
def is_armstrong(n: int) -> bool:
    if n < 0:
        return False
    digits = str(n)
    k = len(digits)
    return n == sum(int(d) ** k for d in digits)


print([n for n in range(1000) if is_armstrong(n)])

In [ ]:
assert is_armstrong(0) and is_armstrong(9)            # single digits: d**1 == d
assert is_armstrong(153) and is_armstrong(370) and is_armstrong(371) and is_armstrong(407)
assert is_armstrong(9474)                              # 4 digits: 9^4 + 4^4 + 7^4 + 4^4
assert not is_armstrong(154) and not is_armstrong(10)
assert not is_armstrong(-153)                          # negatives: define them out
assert [n for n in range(1000) if is_armstrong(n)] == list(range(10)) + [153, 370, 371, 407]
print("armstrong: all tests passed")

**Complexity:** O(d) per number, d = number of digits. The power is the digit *count*, not always 3: that is the usual bug.

### Problem 11 — Factorial (iterative and recursive)

`n! = 1 × 2 × … × n`, with `0! = 1`; negative input raises `ValueError`. Example: `5! = 120`.

In [ ]:
def factorial_iter(n: int) -> int:
    if n < 0:
        raise ValueError("factorial is undefined for negative numbers")
    result = 1
    for k in range(2, n + 1):
        result *= k
    return result


def factorial_rec(n: int) -> int:
    if n < 0:
        raise ValueError("factorial is undefined for negative numbers")
    return 1 if n <= 1 else n * factorial_rec(n - 1)


print(factorial_iter(5), factorial_rec(20))
try:
    factorial_rec(5000)
except RecursionError:
    print(f"factorial_rec(5000) -> RecursionError (limit {sys.getrecursionlimit()} frames)")

In [ ]:
for f in (factorial_iter, factorial_rec):
    assert f(0) == 1 and f(1) == 1                     # the base cases
    assert f(5) == 120
    assert f(20) == 2_432_902_008_176_640_000
    assert all(f(n) == math.factorial(n) for n in range(50))
    assert raises(ValueError, f, -1)                   # negatives
assert factorial_iter(21) > 2**63 - 1                  # would overflow int64; Python ints just grow
assert factorial_iter(5000) == math.factorial(5000)    # iterative has no depth limit
print("factorial: all tests passed")

**Complexity:** O(n) multiplications (on ever-bigger integers). The recursive version also uses O(n) stack and hits Python's recursion limit (1000 frames) near n ≈ 1000; Python has no tail-call optimisation.

### Problem 12 — Fibonacci (naive, memoised, iterative)

`F(0)=0, F(1)=1, F(n)=F(n−1)+F(n−2)`. Example: `F(10) = 55`. Then time naive vs memoised at n = 35.

**In plain English:** the naive recursion recomputes the same values again and again (F(33) is computed twice, F(32) three times, and so on). Memoisation stores each answer the first time, so each F(k) is computed once.

In [ ]:
def fib_naive(n: int) -> int:
    return n if n < 2 else fib_naive(n - 1) + fib_naive(n - 2)


@functools.lru_cache(maxsize=None)                 # same as @functools.cache
def fib_memo(n: int) -> int:
    return n if n < 2 else fib_memo(n - 1) + fib_memo(n - 2)


def fib_iter(n: int) -> int:
    if n < 0:
        raise ValueError("n must be >= 0")
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a


print([fib_iter(n) for n in range(11)])

In [ ]:
first10 = [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
for f in (fib_naive, fib_memo, fib_iter):
    assert [f(n) for n in range(10)] == first10
assert fib_iter(100) == fib_memo(100) == 354_224_848_179_261_915_075
assert raises(ValueError, fib_iter, -1)                # negatives

calls = 0
def fib_counted(n):
    global calls
    calls += 1
    return n if n < 2 else fib_counted(n - 1) + fib_counted(n - 2)

fib_counted(25)
assert calls == 2 * fib_iter(26) - 1                   # naive makes 2·F(n+1) − 1 calls
print(f"naive fib(25) made {calls:,} calls; fib(35) would make {2 * fib_iter(36) - 1:,}")
print("fibonacci: all tests passed")

Now measure it. The naive curve is timed for n = 5..30 and once at n = 35 (this cell takes a few seconds; that *is* the lesson).

In [ ]:
ns = list(range(5, 31))
naive_s, memo_s = [], []
for n in ns:
    t0 = time.perf_counter(); fib_naive(n); naive_s.append(time.perf_counter() - t0)
    fib_memo.cache_clear()
    t0 = time.perf_counter(); fib_memo(n); memo_s.append(time.perf_counter() - t0)

t0 = time.perf_counter(); f35 = fib_naive(35); naive35 = time.perf_counter() - t0
fib_memo.cache_clear()
t0 = time.perf_counter(); fib_memo(35); memo35 = time.perf_counter() - t0
info = fib_memo.cache_info()
print(f"fib(35) = {f35:,}")
print(f"naive   : {naive35:6.2f} s   ({2 * fib_iter(36) - 1:,} calls)")
print(f"memoised: {memo35 * 1e6:6.0f} µs  ({info.misses} computed, {info.hits} cache hits) -> {naive35 / memo35:,.0f}x faster")
assert info.misses == 36 and memo35 * 100 < naive35

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(ns, naive_s, color=ORANGE, lw=2, marker="o", ms=4, label="naive recursion")
ax.plot(ns, memo_s, color=BLUE, lw=2, marker="o", ms=4, label="memoised (lru_cache)")
ax.plot([35], [naive35], marker="*", ms=14, color=ORANGE, ls="none")
ax.annotate(f"n = 35: {naive35:.1f} s", (35, naive35), xytext=(-14, -4), textcoords="offset points",
            ha="right", va="top")
ax.set_yscale("log")
ax.set_ylim(top=naive35 * 5)                     # headroom so the n = 35 point isn't on the edge
ax.set_xlabel("n"); ax.set_ylabel("seconds (log scale)")
ax.set_title("Naive Fibonacci is exponential (a straight line on a log axis); memoised is linear")
ax.legend(loc="upper left")
plt.show()

**Complexity:** naive O(φⁿ) ≈ O(1.618ⁿ) time and O(n) stack; memoised O(n) time and space (still recursive, so still capped by the recursion limit); iterative O(n) time and O(1) space, which is the production answer. The senior follow-up is fast doubling or matrix power: O(log n) multiplications.

### Problem 13 — Prime check and the Sieve of Eratosthenes

`is_prime(n)` by trial division up to √n; `primes_up_to(n)` with the sieve.
Example: `is_prime(97)` → `True`; `primes_up_to(30)` → `[2, 3, 5, 7, 11, 13, 17, 19, 23, 29]`.

**In plain English:** a composite number must have a divisor ≤ √n, so trial division can stop there. To list *many* primes, the sieve crosses out multiples of each prime instead of testing every number separately.

In [ ]:
def is_prime(n: int) -> bool:
    if n < 2:
        return False
    if n % 2 == 0:
        return n == 2
    for d in range(3, math.isqrt(n) + 1, 2):       # odd divisors up to sqrt(n)
        if n % d == 0:
            return False
    return True


def primes_up_to(n: int) -> list[int]:
    """Sieve of Eratosthenes: cross out multiples of each prime, starting at p*p."""
    if n < 2:
        return []
    flags = [True] * (n + 1)
    flags[0] = flags[1] = False
    for p in range(2, math.isqrt(n) + 1):
        if flags[p]:
            flags[p * p :: p] = [False] * len(range(p * p, n + 1, p))   # slice assignment runs in C
    return [i for i, is_p in enumerate(flags) if is_p]


print(is_prime(97), primes_up_to(30))

In [ ]:
assert not any(is_prime(n) for n in (-7, 0, 1, 4, 9, 561))   # negatives, 0, 1, squares, a Carmichael number
assert is_prime(2) and is_prime(3) and is_prime(97)
assert is_prime(7919)                                  # the 1000th prime
assert is_prime(1_000_000_007)                         # a big prime: ~16k trial divisions
assert primes_up_to(-5) == [] and primes_up_to(1) == [] and primes_up_to(2) == [2]
assert primes_up_to(30) == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
assert len(primes_up_to(10_000)) == 1229               # pi(10^4)
assert primes_up_to(2000) == [n for n in range(2001) if is_prime(n)]
print("primes: all tests passed")

In [ ]:
sizes = [25_000, 50_000, 100_000, 200_000]
trial_ms, sieve_ms = [], []
for N in sizes:
    t0 = time.perf_counter(); by_trial = [n for n in range(N + 1) if is_prime(n)]
    trial_ms.append((time.perf_counter() - t0) * 1000)
    t0 = time.perf_counter(); by_sieve = primes_up_to(N)
    sieve_ms.append((time.perf_counter() - t0) * 1000)
    assert by_trial == by_sieve
    print(f"N={N:>7,}: {len(by_sieve):>6,} primes | trial division {trial_ms[-1]:6.0f} ms | sieve {sieve_ms[-1]:5.1f} ms")
assert sieve_ms[-1] * 3 < trial_ms[-1]

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(sizes, trial_ms, color=ORANGE, lw=2, marker="o", label="trial division for every n")
ax.plot(sizes, sieve_ms, color=BLUE, lw=2, marker="o", label="sieve of Eratosthenes")
ax.set_xticks(sizes, [f"{N // 1000}k" for N in sizes])
ax.set_xlabel("N (list all primes ≤ N)"); ax.set_ylabel("milliseconds")
ax.set_title("Listing primes: the sieve does far less work than testing each number")
ax.legend(loc="upper left")
plt.show()

**Complexity:** `is_prime` is O(√n) time, O(1) space. The sieve is O(n log log n) time and O(n) space. For one number use trial division (or Miller–Rabin for huge n); for all primes up to N use the sieve.

### Problem 14 — GCD and LCM (Euclid)

`gcd(a, b) = gcd(b, a mod b)` until b is 0; `lcm(a, b) = |a·b| / gcd(a, b)`.
Example: `gcd(48, 18)` → `6`; `lcm(4, 6)` → `12`.

In [ ]:
def gcd(a: int, b: int) -> int:
    a, b = abs(a), abs(b)
    while b:
        a, b = b, a % b
    return a


def lcm(a: int, b: int) -> int:
    if a == 0 or b == 0:
        return 0
    return abs(a // gcd(a, b) * b)                 # divide first to keep numbers small


print(gcd(48, 18), lcm(4, 6), functools.reduce(gcd, [12, 18, 24]), functools.reduce(lcm, [2, 3, 4]))

In [ ]:
assert gcd(48, 18) == 6 and gcd(18, 48) == 6           # order does not matter
assert gcd(17, 5) == 1                                 # coprime
assert gcd(0, 5) == 5 and gcd(0, 0) == 0               # zeros
assert gcd(-48, 18) == 6 and lcm(-4, 6) == 12          # negatives
assert lcm(4, 6) == 12 and lcm(0, 5) == 0 and lcm(7, 7) == 7
grid = range(-30, 31)
assert all(gcd(a, b) == math.gcd(a, b) and lcm(a, b) == math.lcm(a, b) for a in grid for b in grid)
print("gcd/lcm: all tests passed")

**Complexity:** O(log min(a, b)) steps (the worst case is two consecutive Fibonacci numbers). In real code, `math.gcd` and `math.lcm` take any number of arguments.

### Problem 15 — Number to words (0–999)

Example: `0` → `"zero"`, `42` → `"forty-two"`, `115` → `"one hundred fifteen"`, `999` → `"nine hundred ninety-nine"`. Outside 0–999 → `ValueError`.

In [ ]:
ONES = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten",
        "eleven", "twelve", "thirteen", "fourteen", "fifteen", "sixteen", "seventeen",
        "eighteen", "nineteen"]
TENS = ["", "", "twenty", "thirty", "forty", "fifty", "sixty", "seventy", "eighty", "ninety"]


def number_to_words(n: int) -> str:
    if not 0 <= n <= 999:
        raise ValueError("only 0..999 is supported")
    if n < 20:
        return ONES[n]
    if n < 100:
        tens, ones = divmod(n, 10)
        return TENS[tens] + (f"-{ONES[ones]}" if ones else "")
    hundreds, rest = divmod(n, 100)
    return f"{ONES[hundreds]} hundred" + (f" {number_to_words(rest)}" if rest else "")


print([number_to_words(n) for n in (0, 13, 42, 100, 115, 999)])

In [ ]:
cases = {0: "zero", 7: "seven", 13: "thirteen", 20: "twenty", 21: "twenty-one",
         99: "ninety-nine", 100: "one hundred", 101: "one hundred one",
         110: "one hundred ten", 999: "nine hundred ninety-nine"}
assert all(number_to_words(n) == words for n, words in cases.items())
assert raises(ValueError, number_to_words, -1) and raises(ValueError, number_to_words, 1000)
all_words = [number_to_words(n) for n in range(1000)]
assert len(set(all_words)) == 1000                     # every number gets a distinct phrase
assert not any("  " in w or w.endswith((" ", "-")) for w in all_words)   # no stray separators
print("number_to_words: all tests passed")

**Complexity:** O(1) (the input is bounded). Clarify the style first: British English says "one hundred *and* fifteen". Extending to thousands and millions is the same function applied to each 3-digit group.

## 4 · Lists

### Problem 16 — Max and min without built-ins

Return `(min, max)` in one pass without `min()`, `max()` or `sorted()`; empty input → `ValueError`.
Example: `[3, -1, 7, 7]` → `(-1, 7)`.

In [ ]:
def min_max(iterable):
    it = iter(iterable)                          # works for lists, tuples, generators
    try:
        lo = hi = next(it)
    except StopIteration:
        raise ValueError("min_max() arg is an empty iterable") from None
    for x in it:
        if x < lo:
            lo = x
        elif x > hi:                             # can't be both a new low and a new high
            hi = x
    return lo, hi


print(min_max([3, -1, 7, 7]), min_max(x * x for x in range(-3, 4)))

In [ ]:
assert min_max([5]) == (5, 5)                          # single
assert min_max([3, -1, 7, 7]) == (-1, 7)               # duplicates
assert min_max([-5, -2, -9]) == (-9, -2)               # all negative
assert min_max([2, 2, 2]) == (2, 2)
assert min_max(["pear", "apple", "fig"]) == ("apple", "pear")   # anything comparable
assert min_max(x * x for x in range(-3, 4)) == (0, 9)  # a generator
assert raises(ValueError, min_max, [])                 # empty
for _ in range(200):
    data = [random.randint(-100, 100) for _ in range(random.randint(1, 30))]
    assert min_max(data) == (min(data), max(data))
print("min_max: all tests passed")

**Complexity:** O(n) time, O(1) space, one pass. Follow-up: comparing elements in pairs finds both in about 1.5n comparisons instead of 2n.

### Problem 17 — Second largest

Return the second largest *distinct* value, or `None` if there is none.
Example: `[4, 1, 4, 3]` → `3`; `[5, 5]` → `None`.

In [ ]:
def second_largest_sort(nums):
    distinct = sorted(set(nums), reverse=True)           # O(n log n)
    return distinct[1] if len(distinct) >= 2 else None


def second_largest(nums):
    first = second = None
    for x in nums:
        if first is None or x > first:
            first, second = x, first                     # old champion becomes runner-up
        elif x != first and (second is None or x > second):
            second = x
    return second


print(second_largest([4, 1, 4, 3]), second_largest([5, 5]))

In [ ]:
for f in (second_largest_sort, second_largest):
    assert f([]) is None and f([7]) is None            # empty, single
    assert f([5, 5]) is None                           # duplicates only
    assert f([4, 1, 4, 3]) == 3
    assert f([-2, -5, -1]) == -2                       # negatives
    assert f([1, 2]) == 1 and f([2, 1]) == 1
for _ in range(300):
    data = [random.randint(-20, 20) for _ in range(random.randint(0, 12))]
    assert second_largest(data) == second_largest_sort(data)
print("second_largest: all tests passed")

**Complexity:** one pass is O(n) time and O(1) space; the sort version is O(n log n). Ask first: for `[4, 4, 3]`, is the answer 4 or 3? "Distinct" changes the code.

### Problem 18 — Remove duplicates, preserving order

Example: `[3, 1, 3, 2, 1]` → `[3, 1, 2]`.

In [ ]:
def dedupe_brute(items):
    return [x for i, x in enumerate(items) if x not in items[:i]]   # O(n²): scans the prefix


def dedupe_seen(items):
    seen, out = set(), []
    for x in items:
        if x not in seen:                        # O(1) average membership test
            seen.add(x)
            out.append(x)
    return out


def dedupe_dict(items):
    return list(dict.fromkeys(items))            # dicts keep insertion order (3.7+)


print(dedupe_dict([3, 1, 3, 2, 1]), dedupe_dict("mississippi"))

In [ ]:
for f in (dedupe_brute, dedupe_seen, dedupe_dict):
    assert f([]) == [] and f([1]) == [1]               # empty, single
    assert f([3, 1, 3, 2, 1]) == [3, 1, 2]
    assert f("mississippi") == ["m", "i", "s", "p"]
    assert f([-1, 1, -1]) == [-1, 1]                   # negatives
    assert f([1, True, 1.0]) == [1]                    # 1 == True == 1.0 and they hash alike!
assert raises(TypeError, dedupe_dict, [[1], [1]])      # lists are unhashable
print("dedupe: all tests passed")

**Complexity:** O(n) average time and O(n) space with a set or dict; the brute force is O(n²). `list(set(xs))` also dedupes but loses the order. For unhashable items, dedupe by a key (see [03 · dedupe dicts by a key](03_list_and_dict_problems.ipynb)).

### Problem 19 — Missing number in 1..n

The list holds n−1 distinct numbers from 1..n; find the missing one.
Example: `[3, 7, 1, 2, 8, 4, 5]` → `6` (n = 8).

In [ ]:
def missing_set(nums):
    return (set(range(1, len(nums) + 2)) - set(nums)).pop()      # clear, but O(n) extra space


def missing_sum(nums):
    n = len(nums) + 1
    return n * (n + 1) // 2 - sum(nums)                          # Gauss: 1 + ... + n


def missing_xor(nums):
    acc = 0
    for i, x in enumerate(nums, start=1):
        acc ^= i ^ x                                             # pairs cancel: x ^ x == 0
    return acc ^ (len(nums) + 1)


print(missing_set([3, 7, 1, 2, 8, 4, 5]), missing_sum([3, 7, 1, 2, 8, 4, 5]), missing_xor([3, 7, 1, 2, 8, 4, 5]))

In [ ]:
big = list(range(1, 10_001)); big.remove(4321); random.shuffle(big)
for f in (missing_set, missing_sum, missing_xor):
    assert f([]) == 1                                  # empty: n = 1, so 1 is missing
    assert f([1]) == 2 and f([2]) == 1                 # single
    assert f([3, 7, 1, 2, 8, 4, 5]) == 6
    assert f([2, 3, 4]) == 1 and f([1, 2, 3]) == 4     # missing at either end
    assert f(big) == 4321
print("missing number: all tests passed")

**Complexity:** O(n) time for all three; O(1) extra space for sum and XOR. In fixed-width languages the sum can overflow and XOR cannot, which is why XOR is the follow-up. These tricks assume no duplicates, so validate that or clarify it.

### Problem 20 — Flatten a nested list

Any depth; strings stay whole. Example: `[1, [2, [3, [4]], 5], []]` → `[1, 2, 3, 4, 5]`.

In [ ]:
def flatten(nested):
    out = []
    for item in nested:
        if isinstance(item, (list, tuple)):
            out.extend(flatten(item))
        else:
            out.append(item)
    return out


def iflatten(nested):
    """Generator: lazy, builds no intermediate lists."""
    for item in nested:
        if isinstance(item, (list, tuple)):
            yield from iflatten(item)
        else:
            yield item


def flatten_stack(nested):
    """Explicit stack of iterators: no recursion, so no recursion limit."""
    stack, out = [iter(nested)], []
    while stack:
        for item in stack[-1]:
            if isinstance(item, (list, tuple)):
                stack.append(iter(item))         # descend; resume this level later
                break
            out.append(item)
        else:                                    # loop finished without break: level exhausted
            stack.pop()
    return out


print(flatten([1, [2, [3, [4]], 5], []]), list(iflatten([["ab"], ("cd", ["e"])])))

In [ ]:
deep = [1]
for _ in range(5000):
    deep = [deep]                                      # 5000 levels deep
for f in (flatten, lambda x: list(iflatten(x)), flatten_stack):
    assert f([]) == [] and f([[]]) == [] and f([1]) == [1]   # empty, empty inner, single
    assert f([1, [2, [3, [4]], 5], []]) == [1, 2, 3, 4, 5]
    assert f(["ab", ["cd"]]) == ["ab", "cd"]           # strings are atoms, not lists of chars
    assert f((1, (2, -3))) == [1, 2, -3]               # tuples and negatives
    assert f([None, [None]]) == [None, None]
assert raises(RecursionError, flatten, deep)           # recursion depth = nesting depth
assert flatten_stack(deep) == [1]
print("flatten: all tests passed")

**Complexity:** O(N) time for N items plus sublists; the recursive versions use O(depth) call stack and fail past about 1000 levels, while the explicit stack does not. Trap: testing "is it iterable?" instead of "is it a list?" recurses forever on strings, because `"a"[0]` is `"a"` again.

## Try it yourself

**1.** Reverse the digits of an integer, keeping the sign: `123 → 321`, `-120 → -21`, `0 → 0`.

In [ ]:
# Solution — try it yourself first, then run this
def reverse_int(n: int) -> int:
    sign = -1 if n < 0 else 1
    return sign * int(str(abs(n))[::-1])


assert reverse_int(123) == 321
assert reverse_int(-120) == -21                  # trailing zeros vanish
assert reverse_int(0) == 0 and reverse_int(7) == 7
print("reverse_int:", {n: reverse_int(n) for n in (123, -120, 0, 1_000_000)})

**2.** Digital root: keep summing the digits until one digit is left (`9875 → 29 → 11 → 2`). Write it with a loop (reuse `digit_sum_math`), then find the O(1) formula and check the two against each other.

In [ ]:
# Solution — try it yourself first, then run this
def digital_root_loop(n: int) -> int:
    n = abs(n)
    while n >= 10:
        n = digit_sum_math(n)
    return n


def digital_root(n: int) -> int:
    n = abs(n)
    return 0 if n == 0 else 1 + (n - 1) % 9      # "casting out nines"


assert digital_root_loop(9875) == digital_root(9875) == 2
assert all(digital_root_loop(n) == digital_root(n) for n in range(-1000, 20_000))
print("digital roots of 0..20:", [digital_root(n) for n in range(21)])

**3.** A perfect number equals the sum of its proper divisors (`6 = 1 + 2 + 3`). List them below 10,000. Hint: divisors come in pairs `(d, n // d)`, so you only need to loop to √n.

In [ ]:
# Solution — try it yourself first, then run this
def divisor_sum(n: int) -> int:
    """Sum of proper divisors (excluding n itself)."""
    if n < 2:
        return 0
    total = 1
    for d in range(2, math.isqrt(n) + 1):
        if n % d == 0:
            total += d
            if d != n // d:                      # don't count a square root twice
                total += n // d
    return total


perfect = [n for n in range(2, 10_000) if divisor_sum(n) == n]
print("perfect numbers below 10,000:", perfect)
assert perfect == [6, 28, 496, 8128]

## Say it in an interview

**The 30-second process answer** (use it on every warm-up): *"Let me restate the problem and check the edge cases: empty input, one element, duplicates, negatives, non-ASCII text. The brute force is ___ at O(__). We can do better with ___ because ___. I'll code that, then walk through the edge cases."*

| Problem | What to say | Trap to mention |
|---|---|---|
| Reverse a string | `s[::-1]`, O(n); a copy is unavoidable because strings are immutable | flags and accents are several code points |
| Palindrome | keep `isalnum()` chars, `casefold()`, compare with the reverse; two pointers for O(1) space | `É` ≠ `e` unless you strip accents |
| Anagram | `Counter(a) == Counter(b)`, O(n) | `set()` forgets duplicates |
| Fibonacci | loop with two variables, O(n) time / O(1) space | naive makes 2·F(n+1)−1 calls (~30 million at n = 35) |
| Primes | trial division to √n for one number; the sieve for all up to N | 0 and 1 are not prime; cross out from p² |
| Missing number | Gauss sum or XOR, O(1) space | sum overflows in fixed-width languages |
| Dedupe, keep order | `list(dict.fromkeys(xs))` | `1`, `True` and `1.0` are the same key |
| Flatten | recursion or `yield from`; an explicit stack for very deep input | strings are iterable |

**Key numbers:** Python's recursion limit is 1,000 frames; Python ints never overflow (`20!` fits in int64, `21!` does not); there are 1,229 primes below 10,000; memoised fib(35) computes 36 values instead of ~30 million calls.

**Follow-ups to expect:** "Now without the built-in." "What if the input is a stream that doesn't fit in memory?" (a generator). "What is the space complexity?" "How would you test it?" (you just did: empty, single, duplicates, negatives, unicode).

## Next

- [02 · String problems](02_string_problems.ipynb): the same drill for text (sliding windows, stacks, regex, unicode).
- [03 · List and dict problems](03_list_and_dict_problems.ipynb): hashing, two pointers, counting.
- [Big-O and timing](../03_dsa/01_big_o_and_timing.ipynb): why the timing curves above have the shapes they do.
- [Recursion and backtracking](../03_dsa/08_recursion_and_backtracking.ipynb) and [Dynamic programming](../03_dsa/12_dynamic_programming.ipynb): where memoised Fibonacci leads.
- [Python gotchas interviewers love](../01_python_basics/16_python_gotchas_interviewers_love.ipynb)
- Theory: [python_basics course](../../python_basics/index.html) · [dsa_basics course](../../dsa_basics/index.html)